# DeFi exploration — live time-series

Reads `data/defi.duckdb` (Hyperliquid perps, Coinbase/Kraken spot, Polymarket crypto books) and visualizes the substrate we're collecting toward time-series prediction. Sparse cells fill in as the 5-min cron accumulates.

In [ ]:
import sys
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')

def repo_root():
    r = Path.cwd()
    while not (r/'data').exists() and r != r.parent: r = r.parent
    return r
ROOT = repo_root()
con = duckdb.connect(str(ROOT/'data'/'defi.duckdb'), read_only=True)
print('connected:', ROOT/'data'/'defi.duckdb')

## Coverage

In [ ]:
for t in ['hl_ctx','hl_book','cex_spot','pm_book']:
    n,mn,mx = con.execute(f'select count(*), min(captured_at), max(captured_at) from {t}').fetchone()
    print(f'{t:9} rows={n:>7}  {mn} -> {mx}')

## Hyperliquid funding & open interest (latest snapshot, ~230 coins)

In [ ]:
ctx = con.execute('''
  WITH latest AS (SELECT max(captured_at) m FROM hl_ctx)
  SELECT coin, funding, open_interest, mark_px, day_ntl_vlm
  FROM hl_ctx, latest WHERE captured_at = latest.m''').df()
top = ctx.sort_values('open_interest', ascending=False).head(20).copy()
top['funding_apr'] = top['funding']*24*365*100   # HL funding is hourly
fig, ax = plt.subplots(1,2, figsize=(13,6))
sns.barplot(data=top, y='coin', x='open_interest', ax=ax[0], color='steelblue')
ax[0].set_title('Open interest (top 20 by OI)')
sns.barplot(data=top.sort_values('funding_apr'), y='coin', x='funding_apr', ax=ax[1], color='indianred')
ax[1].set_title('Funding rate (annualized %)'); plt.tight_layout(); plt.show()

## Basis: Hyperliquid mark vs Coinbase spot

ASOF-joined so slightly-offset capture times still align. A basis much larger than the inter-CEX spread is a real venue dislocation.

In [ ]:
basis = con.execute('''
  WITH cb AS (SELECT asset, captured_at, last FROM cex_spot WHERE venue='coinbase')
  SELECT h.captured_at, h.coin AS asset, h.mark_px AS hl_mark, cb.last AS cex,
         h.mark_px - cb.last AS basis
  FROM hl_ctx h ASOF JOIN cb ON h.coin = cb.asset AND h.captured_at >= cb.captured_at
  ORDER BY h.captured_at''').df()
assets = ['BTC','ETH','SOL']
if basis['captured_at'].nunique() >= 2:
    fig, ax = plt.subplots(figsize=(11,5))
    for a in assets:
        d = basis[basis.asset==a]
        if len(d): ax.plot(d.captured_at, d.basis, marker='o', label=a)
    ax.axhline(0, color='k', lw=.6); ax.set_ylabel('HL mark - CEX last'); ax.legend(); ax.set_title('Basis over time'); plt.show()
else:
    print('Only', basis['captured_at'].nunique(), 'snapshot(s) — basis snapshot below; time series fills as cron runs.')
    display(basis[basis.asset.isin(assets)][['asset','hl_mark','cex','basis']])

## Polymarket crypto markets (latest mids)

In [ ]:
pm = con.execute('''
  WITH latest AS (SELECT token_id, max(captured_at) m FROM pm_book GROUP BY token_id)
  SELECT p.question, p.outcome, p.mid, p.best_bid, p.best_ask, p.liquidity_num
  FROM pm_book p JOIN latest l ON p.token_id=l.token_id AND p.captured_at=l.m
  ORDER BY p.liquidity_num DESC''').df()
print(len(pm), 'token rows'); pm.head(20)

## CEX↔DEX lead-lag

Reuses `research/defi/analyze_leadlag.py`. Positive lag ⇒ Hyperliquid leads. Needs ~22 grid points; reports insufficient until then.

In [ ]:
sys.path.insert(0, str(ROOT/'research'/'defi'))
from analyze_leadlag import aligned_series, lead_lag, MIN_RETURNS
for a in ['BTC','ETH','SOL']:
    hl, cx = aligned_series(con, a, 5, 'coinbase')
    if len(hl) < MIN_RETURNS + 2:
        print(f'{a}: {len(hl)} grid pts — need {MIN_RETURNS+2}; let the cron accumulate.'); continue
    lag, corr, _ = lead_lag(hl, cx, 6)
    who = 'HL leads' if lag>0 else ('CEX leads' if lag<0 else 'synchronous')
    print(f'{a}: best lag {lag:+d} ({lag*5:+d}m) corr {corr:+.3f} -> {who}')

## First baseline: does the basis predict the next CEX move?

The simplest predictive question — sign agreement between current basis and the next-period CEX return. A real signal would beat 50%.

In [ ]:
rows = []
for a in ['BTC','ETH','SOL','DOGE','XRP']:
    d = basis[basis.asset==a].sort_values('captured_at')
    if len(d) < 12: continue
    cex = d['cex'].to_numpy(); b = d['basis'].to_numpy()[:-1]
    fwd_ret = np.diff(cex)
    hit = np.mean(np.sign(b) == np.sign(fwd_ret))
    rows.append((a, len(d), round(float(hit),3)))
if rows:
    display(pd.DataFrame(rows, columns=['asset','points','basis_predicts_next_move_hitrate']))
else:
    print('Not enough history yet (need >=12 points/asset). This is the cell to watch as data grows.')